In this notebook, I'll re-implement a transformer-based machine translation algorithm. This will be based on Chapter 11 of the D2L.AI textbook. In doing so, I'll try to do everything from memory. This means not consulting the D2L.AI textbook or the code I wrote previously.

# 1. Get Dataset

We'll use the FraEngMT dataset. We've already written code to extract batches from this dataset in `utils.data` and won't rewrite the cleaning, tokenization, and extraction algorithms from scratch.

In [1]:
import os
import sys
print(os.getcwd())
sys.path.append(os.path.join(os.getcwd(), '..'))
from utils.data import FraEngMTDataset, get_data_mt

/Users/nickmcgreivy/code/ml-practice/ml-experiments/natural_language_processing


In [2]:
batch_size = 8
num_steps = 10
num_train = 1024
num_val = 512
train_dl, val_dl, src_vocab, tgt_vocab = get_data_mt(batch_size, num_steps, num_train, num_val)

The dataloaders return 4 arrays: `src`, `tgt`, `src_valid_len`, `tgt_label`

In [3]:
for src, tgt, src_valid_len, tgt_label in train_dl:
    print(src.shape, tgt.shape, src_valid_len.shape, tgt_label.shape)
    print(src[0], tgt[0], src_valid_len[0], tgt_label[0])
    print(src_vocab.to_tokens(src[0]))
    print(tgt_vocab.to_tokens(tgt[0]))
    print(tgt_vocab.to_tokens(tgt_label[0]))
    print(src_valid_len[0])
    break

torch.Size([8, 10]) torch.Size([8, 10]) torch.Size([8]) torch.Size([8, 10])
tensor([293,  41,   2,   4,   5,   5,   5,   5,   5,   5]) tensor([  3, 348,  63,   2,   4,   5,   5,   5,   5,   5]) tensor(4) tensor([348,  63,   2,   4,   5,   5,   5,   5,   5,   5])
['stay', 'calm', '.', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
['<bos>', 'reste', 'calme', '.', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
['reste', 'calme', '.', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
tensor(4)


# 2. Create Encoder-Decoder Abstract Class

Our model will use an encoder-decoder architecture. The encoder will return a tuple of outputs, and then those outputs will be fed as inputs to the decoder. 

These will subclass `models.Module`, which allows for my pre-implemented plotting routines during training.

In [4]:
from abc import ABC
from utils.models import Module

In [5]:
class Encoder(Module, ABC):
    def __init__(self):
        pass

    def forward(self):
        pass

In [6]:
class Decoder(Module, ABC):
    def __init__(self):
        pass

    def init_state(self):
        pass

    def forward(self):
        pass

In [7]:
class EncoderDecoder(Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, tgt, *args):
        enc_state = self.encoder(src, *args)
        dec_init_state = self.decoder.init_state(enc_state, *args)
        return self.decoder(tgt, dec_init_state, *args)

# 3. Implement Transformer Encoder Block

In [8]:
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
class MultiHeadAttention(Module):
    def __init__(self, num_hiddens, num_heads, dropout=0.5):
        super().__init__()
        pass
    
    def forward(self):
        pass

In [ ]:
class PositionWiseFFN(Module):
    def __init__(self, num_hiddens, ffn_num_hiddens, dropout=0.5):
        super().__init__()
        pass
        
    def forward(self, X):
        pass

In [ ]:
class TransformerEncoderBlock(Module):
    def __init__(self, num_hiddens, num_heads, ffn_num_hiddens, dropout=0.5):
        super().__init__()
        self.ln1 = nn.LayerNorm(num_hiddens)
        self.attention = MultiHeadAttention(num_hiddens, num_heads, dropout=dropout)
        self.ln2 = nn.LayerNorm(num_hiddens)
        self.ffn = PositionWiseFFN(num_hiddens, ffn_num_hiddens, dropout=dropout)

    def forward(self, X, valid_len):
        Y = X + self.attention(self.ln1(X))
        return Y + self.ffn(self.ln2(Y))

# 4. Write Transformer-Based Encoder

We haven't implemented the details of the transformer block yet. We will implement those details after creating the structure of the model.

In [ ]:
class TransformerEncoder(Encoder):
    def __init__(self, num_hiddens, num_blks, num_heads, ffn_num_hiddens, src_vocab, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(len(src_vocab), num_hiddens)
        self.blks = nn.ModuleList()
        for i in range(num_blks):
            self.blks.append(TransformerEncoderBlock(num_hiddens, num_heads, ffn_num_hiddens, dropout=dropout))
        
    def forward(self, src, src_valid_len):
        X = self.embedding(src)
        for blk in self.blks:
            X = blk(X, src_valid_len)
        return X

# 5. Write Transfor-Based Decoder Block

In [ ]:
class TransformerDecoderBlock(Module):
    def __init__(self, num_hiddens, num_heads, ffn_num_hiddens, dropout=0.5):
        super().__init__()
        self.ln1 = nn.LayerNorm(num_hiddens)
        self.attention1 = MultiHeadAttention(num_hiddens, num_heads, dropout=dropout)
        self.ln2 = nn.LayerNorm(num_hiddens)
        self.attention2 = MultiHeadAttention(num_hiddens, num_heads, dropout=dropout)
        self.ln3 = nn.LayerNorm(num_hiddens)
        self.ffn = PositionWiseFFN(num_hiddens, ffn_num_hiddens, dropout=dropout)

    def forward(self, X, dec_init_state, src_valid_len, tgt_valid_len):
        Xn = self.ln1(X)
        Y1 = X + self.attention1(Xn, Xn, Xn, tgt_valid_len)
        Y1n = self.ln2(Y1)
        Y2 = Y1 + self.attention2(Y1n, dec_init_state, dec_init_state, src_valid_len)
        return Y2 + self.ffn(self.ln3(Y2))

# 6. Write Transformer-Based Decoder

The decoder takes the output of the encoder, passes it to `init_state`, then passes `tgt`, `dec_init_state`, and `src_valid_len` to the decoder. `tgt` is passed through an embedding layer, and both are passed to a sequence of `TransformerDecoderBlock`.

In [ ]:
class TransformerDecoder(Module):
    def __init__(self, num_hiddens, num_blks, num_heads, ffn_num_hiddens, tgt_vocab, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(len(tgt_vocab), num_hiddens)
        self.blks = nn.ModuleList()
        for i in range(num_blks):
            self.blks.append(TransformerDecoderBlock(num_hiddens, num_heads, ffn_num_hiddens, dropout=dropout))
        self.linear_out = nn.Linear(num_hiddens, len(tgt_vocab))
    
    def init_state(self, enc_output, src_valid_len):
        return enc_output
    
    def forward(self, tgt, dec_init_state, src_valid_len):
        X = self.embedding(tgt)
        tgt_valid_len = ...
        for blk in self.blks:
            X = blk(X, dec_init_state, src_valid_len, tgt_valid_len)
        return self.linear_out(X)

# 7. Train model

In [ ]:
from utils.train import fit_mt

In [ ]:
lr = 1e-3
num_hiddens = 64
ffn_num_hiddens = 32
num_blks = 4
num_heads = 8
num_epochs = 2

In [ ]:
encoder = TransformerEncoder(num_hiddens, num_blks, num_heads, ffn_num_hiddens, src_vocab)
decoder = TransformerDecoder(num_hiddens, num_blks, num_heads, ffn_num_hiddens, tgt_vocab)
model = EncoderDecoder(encoder, decoder)
opt = torch.optim.Adam(model.parameters(), lr=lr)

In [1]:
def masked_loss_fn_mt(tgt, tgt_label, tgt_vocab):
    loss = F.cross_entropy(tgt, tgt_label, reduce='none')
    mask = (tgt_label != tgt_vocab['<pad>']).type(torch.float)
    return torch.mean(loss * mask)

loss_fn_mt = lambda tgt, tgt_label: masked_loss_fn_mt(tgt, tgt_label, tgt_vocab)

In [ ]:
fit_mt(train_dl, val_dl, model, opt, num_epochs=num_epochs, device='cpu', 
       id='TestScratchTransformerMachineTranslation', 
       loss_fn=loss_fn_mt)